# **Recombination Analysis - Gubbins**

## **Tool Information**

- **Tool:** Gubbins (v2.3.4)  
- **Input:** Core genome alignment (FASTA)  
- **Organism:** *Acinetobacter baumannii*  
- **Analysis type:** Recombination detection and phylogenetic correction  

Gubbins (Genealogies Unbiased By recomBinations In Nucleotide Sequences) is a tool used to detect regions of homologous recombination in bacterial whole-genome alignments.

It iteratively identifies recombination events and removes them to generate a refined alignment containing only vertically inherited (clonal) regions. This improves the accuracy of phylogenetic inference by eliminating misleading signals introduced by recombination.

This analysis was performed to obtain a recombination-free core genome alignment for robust phylogenetic reconstruction.

# **Create a Dedicated Environment**

We create a dedicated conda environment to isolate Gubbins and its dependencies from other tools. This ensures a stable and reproducible setup for recombination and phylogenetic analyses.

In [ ]:
%%bash

conda create -n gubbins_aba python=3.10 -y

# **Install Gubbins using mamba**

We install Gubbins using mamba for efficient dependency resolution and faster installation. The tool is installed from Bioconda along with required dependencies from conda-forge.

In [ ]:
%%bash

# Initialize conda
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate environment
conda activate gubbins_aba

# Install mamba
conda install -c conda-forge mamba -y

# Install Gubbins
mamba install -c bioconda -c conda-forge gubbins --channel-priority flexible -y

# **Verify Installation**

We verify that Gubbins is installed correctly by checking the tool version. This confirms that the installation was successful and the tool is ready for recombination analysis.

In [4]:
%%bash
source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate gubbins

echo "[INFO] Gubbins version:"
run_gubbins.py --version

[INFO] Gubbins version:
2.3.4


## **Accepted Input Types**

Gubbins can operate on different types of genome alignments, provided they are in FASTA format and properly aligned.

#### 1. SNP-based Pseudogenome Alignment (Used in this study)

- Generated from reference-based SNP calling workflows (like snp_phylogeny which can be found in a separate notebook in the tools repo) 
- Represents genome-wide variation  
- High resolution for phylogenetic and recombination analysis  

Example: `aligned_pseudogenome.fas`

#### 2. Core Genome Alignment (Panaroo / Roary output)

- Derived from pangenome analysis  
- Contains only conserved (core) genes  
- Commonly used in gene-based phylogenetic studies  

Example: `core_alignment.aln`

#### 3. Whole Genome Alignment (De novo alignment tools)

- Generated using tools such as MAFFT or Parsnp  
- May include both core and accessory regions  
- Useful when reference-based mapping is not performed  

### General Input Requirements

Regardless of the input type, the alignment must satisfy the following:

- All sequences must be of equal length  
- Alignment must be positionally consistent  
- Each sequence corresponds to a single genome/isolate  
- Input should be free from major alignment errors  

### Input Selection Rationale

In this study, a SNP-based pseudogenome alignment was used because:

- It captures genome-wide variation  
- Provides higher resolution compared to core gene alignments  
- Improves detection of recombination events across the genome  


## **Execution Strategy**

Gubbins was used to detect homologous recombination events and infer a recombination-filtered phylogeny from the SNP-based Pseudogenome Alignment. The analysis was performed using a maximum-likelihood framework, with recombination regions iteratively identified and masked.

Gubbins was run using 12 threads. All output files were written to a dedicated results directory for downstream analysis.

The workflow includes:

- Identification of recombination regions  
- Masking of recombinant segments  
- Iterative phylogenetic tree construction  
- Generation of recombination-corrected alignment  

Multiple iterations are performed until convergence is reached, ensuring accurate detection of recombination events.

In [ ]:
%%bash
source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate gubbins

# Define input and output paths
INPUT=/data/internship_data/nidhi/aba/new_output/snp_phylogeny/ref_2/pseudogenomes/aligned_pseudogenome.fas
OUTDIR=/data/internship_data/nidhi/aba/new_output/gubbins_output

# Create output directory
mkdir -p "$OUTDIR"
cd "$OUTDIR"

# Run Gubbins
run_gubbins.py \
  --prefix aba_gubbins \
  --threads 32 \
  --tree_builder fasttree \
  "$INPUT"

## **Output Files**

Gubbins generates multiple output files associated with recombination
detection and phylogenetic reconstruction. Important outputs include:

- `aba_gubbins.filtered_polymorphic_sites.fasta`  
  Recombination-filtered core SNP alignment used for downstream phylogenetic
  analysis.

- `aba_gubbins.filtered_polymorphic_sites.phylip`  
  PHYLIP-formatted SNP alignment for phylogenetic software compatibility.

- `aba_gubbins.final_tree.tre`  
  Final recombination-corrected phylogenetic tree in Newick format.

- `aba_gubbins.node_labelled.final_tree.tre`  
  Final phylogenetic tree with internal node labels.

- `aba_gubbins.recombination_predictions.gff`  
  Genomic coordinates of predicted recombination regions in GFF format.

- `aba_gubbins.recombination_predictions.embl`  
  EMBL-formatted recombination prediction annotations.

- `aba_gubbins.per_branch_statistics.csv`  
  Summary statistics describing SNP and recombination events across branches.

- `aba_gubbins.summary_of_snp_distribution.vcf`  
  Variant Call Format (VCF) file summarizing SNP distribution across isolates.

- `aba_gubbins.branch_base_reconstruction.embl`  
  Reconstructed ancestral sequence information in EMBL format.

- `aligned_pseudogenome.fas.iteration_*`  
  Intermediate alignment files generated during iterative recombination
  detection.

- `aligned_pseudogenome.fas.seq.joint.txt`  
  Joint sequence alignment information generated during analysis.

In [10]:
%%bash

ls /data/internship_data/nidhi/aba/output/gubbins_output

aba_gubbins.branch_base_reconstruction.embl
aba_gubbins.filtered_polymorphic_sites.fasta
aba_gubbins.filtered_polymorphic_sites.phylip
aba_gubbins.final_tree.tre
aba_gubbins.nobranch.tre
aba_gubbins.node_labelled.final_tree.tre
aba_gubbins.per_branch_statistics.csv
aba_gubbins.recombination_predictions.embl
aba_gubbins.recombination_predictions.gff
aba_gubbins.summary_of_snp_distribution.vcf
core_gene_alignment.aln.seq.joint.txt
test_10.tre


## **Citation**

Croucher NJ, Page AJ, Connor TR,
Delaney AJ, Keane JA, Bentley SD,
Parkhill J, Harris SR.

Rapid phylogenetic analysis of large samples of recombinant
bacterial whole genome sequences using Gubbins.

Nucleic Acids Research.
2015;43(3):e15.

https://doi.org/10.1093/nar/gku1196